In [7]:
import open3d as o3d
import numpy as np
from pathlib import Path



path = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/inference/6f8f/Backbone_2/map_01")

In [8]:
# Load point cloud data from .npy file
point_cloud = np.load(path / "atom_cloud.npy")
voxels = np.load(path / "voxel_centers.npy")

In [9]:

alpha = 2
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(point_cloud)

mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
area = mesh.get_surface_area()
scene = o3d.t.geometry.RaycastingScene()
mesh_id = scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh))

# === 3. Prepare voxel centers as query points ===
# voxels: (N,3) numpy array of voxel centers

# === 4. Cast rays in +X direction (can choose other directions too) ===
rays = np.concatenate([
    voxels.astype(np.float32),               # origins: voxel centers
    np.tile(np.array([[1,0,0]], dtype=np.float32), (voxels.shape[0],1))  # directions: +X
], axis=1)

# Convert to Open3D Tensor
rays_o3d = o3d.core.Tensor(rays, dtype=o3d.core.Dtype.Float32)

# === 5. Count intersections ===
counts = scene.count_intersections(rays_o3d).numpy()

# === 6. Apply even-odd rule to determine inside/outside ===
# (Strictly, odd counts mean inside under standard ray casting parity rule)
inside_mask = (counts % 2 == 1)

## === 7. Separate inside and outside voxel centers ===
#voxels_inside = voxels[inside_mask]

exterior_voxels = voxels[inside_mask == False]
interior_voxels = voxels[inside_mask == True]


In [10]:

pc_main = o3d.geometry.PointCloud()
pc_main.points = o3d.utility.Vector3dVector(point_cloud)
pc_main.paint_uniform_color([0.1, 0.4, 1.0])  # bluish

# Interior voxels (red, smaller points)
pc_vox = o3d.geometry.PointCloud()
pc_vox.points = o3d.utility.Vector3dVector(interior_voxels)
pc_vox.paint_uniform_color([1.0, 0.1, 0.1])   # red

# Optional coordinate frame (for orientation reference)
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)

# Axis-aligned bounding box (black edges)
bbox = pc_main.get_axis_aligned_bounding_box()
bbox.color = (0, 0, 0)

# ─────────────────────────────────────────────────────────────
# Visualization
# ─────────────────────────────────────────────────────────────

o3d.visualization.draw_geometries(
    [pc_main, pc_vox, frame, bbox],
    zoom=0.3412,
    front=[0.4257, -0.2125, -0.8795],
    lookat=[2.6172, 2.0475, 1.532],
    up=[-0.0694, -0.9768, 0.2024]
)